In [1]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint


TYPE_RENAME = {
    "DistractorNegation": "distractor_negation",
    "DistractorLexicalEdit": "distractor_lexical",
    "DistractorAssociative0": "distractor_associative",
}


def open_and_show_json(path: Path):
    """Open a JSON file and print basic info."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"loaded items: {len(data)}")
    if data:
        print("\nfirst item keys:")
        print(list(data[0].keys()))

        print("\nexample options:")
        pprint(data[0].get("options", {}))

        print("\nexample item meta_general:")
        pprint(data[0].get("meta_general", {}))

        print("\nexample item meta_special:")
        pprint(data[0].get("meta_special", {}))

    return data


def convert_options_implicature(old_options: dict) -> tuple[list[dict], str, str]:
    """
    Convert old implicature options dict into unified options list.

    Returns:
        - converted options list
        - correct option label
        - implicature_paraphrased text (to be stored in metadata)
    """
    labels = ["A", "B", "C", "D"]

    # we keep only ONE correct option in MCQ
    # paraphrased version is preserved separately in metadata
    order = [
        "implicature_true",
        "DistractorNegation",
        "DistractorLexicalEdit",
        "DistractorAssociative0",
    ]

    converted = []
    for label, key in zip(labels, order):
        if key not in old_options:
            continue

        if key == "implicature_true":
            new_type = "implicature_true"
        else:
            new_type = TYPE_RENAME[key]

        converted.append({
            "label": label,
            "type": new_type,
            "text": old_options[key],
        })

    correct_option = next(
        opt["label"] for opt in converted if opt["type"] == "implicature_true"
    )

    implicature_paraphrased = old_options.get("implicature_paraphrased", "")

    return converted, correct_option, implicature_paraphrased


def convert_item_implicature(item: dict) -> dict:
    """Convert one old implicature item to the unified schema."""
    new_options, correct_option, implicature_paraphrased = convert_options_implicature(
        item["options"]
    )

    meta_general = item.get("meta_general", {}) or {}
    meta_special = item.get("meta_special", {}) or {}

    creation_method = (
        meta_general.get("creation_method")
        or meta_general.get("extraction_method")
        or meta_general.get("source")
        or ""
    )

    # optional normalization
    if creation_method == "cql_query":
        creation_method = "corpus_mining"

    metadata = {
        "corpus": meta_general.get("corpus", ""),
        "genre": meta_general.get("genre", ""),
        "trigger": meta_special.get("trigger", ""),
        "negation": meta_special.get("negation", ""),
        # preserve extra useful fields here
        "subtype": meta_special.get("subtype", ""),
        "note": meta_general.get("note", ""),
        "implicature_paraphrased": implicature_paraphrased,
    }

    # remove empty metadata fields
    metadata = {k: v for k, v in metadata.items() if v not in ("", None, [], {})}

    return {
        "id": item["id"],
        "creation_method": creation_method,
        "phenomenon": item["phenomenon"],
        "category": item["category"],
        "context": item["context"],
        "utterance": item["utterance"],
        "question": item.get("question", ""),
        "options": new_options,
        "correct_option": correct_option,
        "metadata": metadata,
    }


def merge_json_lists(input_paths: list[Path]) -> list[dict]:
    """Load multiple JSON files, each containing a list of items, and merge them."""
    merged = []

    for path in input_paths:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            raise ValueError(f"{path} does not contain a JSON list.")

        merged.extend(data)

    return merged


def convert_implicature_files(input_paths: list[Path], output_path: Path) -> list[dict]:
    """
    Merge multiple implicature JSON files, convert all items, and save result.
    """
    merged = merge_json_lists(input_paths)
    converted = [convert_item_implicature(item) for item in merged]

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(converted, f, ensure_ascii=False, indent=2)

    print(f"saved {len(converted)} converted items to {output_path}")
    return converted

In [2]:
from pathlib import Path

input_paths = [
    Path(r"data\implicature\scalar\quantifier_scalar_implicature\qa\implicature-quant-qa-25.json"),
    Path(r"data\implicature\scalar\quantifier_scalar_implicature\declarative\implicature-quant-declarative-25.json"),
    Path(r"data\implicature\scalar\frequency_scalar_implicature\qa\implicature-freq-qa-25.json"),
    Path(r"data\implicature\scalar\frequency_scalar_implicature\declarative\implicature-freq-declarative-25.json")
]

output_path = Path("data/implicature/eval/scalars_eval_100.json")

convert_implicature_files(input_paths, output_path)

saved 100 converted items to data\implicature\eval\scalars_eval_100.json


[{'id': 'implicature-scalar-quant-qa-1',
  'creation_method': 'corpus_mining',
  'phenomenon': 'implicature',
  'category': 'scalar-quantifier',
  'context': 'Mluvčí cituje radikální názor z internetu a následně s kamarádem rozebírá, jak se věci ve skutečnosti mají.',
  'utterance': 'B: Nějaký chlap mi na sociální sítě napsal, že jsou všechny ženy lhářky. \nA: Všechny ženy?\nB: Fajn, řekněme, že některé jsou lhářky.',
  'question': '',
  'options': [{'label': 'A',
    'type': 'implicature_true',
    'text': 'Ne všechny ženy jsou lhářky.'},
   {'label': 'B',
    'type': 'distractor_negation',
    'text': 'Všechny ženy jsou lhářky.'},
   {'label': 'C',
    'type': 'distractor_lexical',
    'text': 'Některé lhářky mají na internetu víc profilů.'},
   {'label': 'D',
    'type': 'distractor_associative',
    'text': 'Uživatelé internetu se snaží lháře v diskusích nahlásit.'}],
  'correct_option': 'A',
  'metadata': {'corpus': 'OpenSubtitles2023',
   'subtype': 'qa',
   'implicature_paraphra